# Word-typography check (audit, 2026-11-21)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohaaa825/Skolfie/blob/main/notebooks/typography_check.ipynb)

The independent audit found that our AI essays and our human student essays differ in typography:
most Gemini essays use curly quotes and em dashes, the human student essays never do. A model can
learn "curly quote = AI", and then every student who types in Word or Google Docs (which turn
`"` into `“ ”` and `--` into `—`) looks more like AI.

This notebook measures it on the trained models. It takes the student essays of the test split
(PERSUADE human essays and our fresh AI essays, full length) and scores them three ways:

* `clean`: as they are;
* `word_typography`: what Word's autocorrect does (curly quotes and apostrophes, en / em dashes, …);
* `plain_typography`: the reverse (straight quotes, `-`, `...`).

If the false-positive rate on human essays jumps under `word_typography`, or AI recall drops
under `plain_typography`, the model leans on typography. Models trained from 2026-11-21 on fold
typography before scoring (normaliser version 2), so for them all three should give the same score.

Runtime: **GPU** (T4 or L4), about 15-30 minutes. Building the text variants needs no GPU, but
scoring with the DeBERTa and Qwen models does.


## 1. Settings

In [ ]:
DATA_DIR = "/content/drive/MyDrive/skolfie/data/processed_v3"
REPORTS_DIR = "/content/drive/MyDrive/skolfie/reports_v3"
MODELS_DIR = "/content/drive/MyDrive/skolfie/models_v3"
MODEL = "stacked"          # the product model; "transformer" or "features" also work

REPO = "mohaaa825/Skolfie"
BRANCH = "main"

SOURCES = "persuade,fresh"  # student essays only: PERSUADE (human) and our AI essays
N_TEXTS = 1000              # up to this many AI and this many human essays (all there are)


## 2. Check the GPU, mount Drive, clone and install

In [ ]:
import os
import subprocess

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU; scoring with the DeBERTa and Qwen models will be very slow.")

from google.colab import drive

drive.mount("/content/drive")
OUT_DIR = f"{REPORTS_DIR}/typography"
os.makedirs(OUT_DIR, exist_ok=True)
for path in (f"{DATA_DIR}/corpus.parquet", f"{DATA_DIR}/splits.parquet",
             f"{MODELS_DIR}/{MODEL}.joblib"):
    assert os.path.exists(path), f"missing {path}"


def secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


def git(*args, check=True):
    """Run git; a GITHUB_TOKEN secret reaches it through a credential helper that reads it from
    the environment, so the token never appears in a URL, a command line, an error message or
    .git/config."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    auth = []
    token = secret("GITHUB_TOKEN")
    if token:
        env["SKOLFIE_GITHUB_TOKEN"] = token
        auth = ["-c", "credential.helper=", "-c",
                "credential.helper=!f() { test \"$1\" = get && echo username=x-access-token "
                "&& echo \"password=$SKOLFIE_GITHUB_TOKEN\"; }; f"]
    return subprocess.run(["git", *auth, *args], env=env, check=check)


url = f"https://github.com/{REPO}.git"
if os.path.isdir("/content/Skolfie/.git"):
    # Clones made by older versions of this notebook kept the token in the remote URL.
    git("-C", "/content/Skolfie", "remote", "set-url", "origin", url)
    git("-C", "/content/Skolfie", "pull", "--ff-only")
else:
    git("clone", "--depth", "1", "--branch", BRANCH, url, "/content/Skolfie")

%cd /content/Skolfie
!git log --oneline -1
!pip install -q -e ".[features]"

## 3. Build the three versions of each essay (CPU work, a few seconds)

In [ ]:
import sys

env = dict(os.environ)
hf_token = secret("HF_TOKEN")
if hf_token:
    env["HF_TOKEN"] = hf_token


def run(cmd):
    """Run a command and stream its output into the notebook as it happens."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env=env)
    for line in proc.stdout:
        sys.stdout.write(line)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


run(["python", "-u", "scripts/make_attacks.py",
     "--corpus", f"{DATA_DIR}/corpus.parquet",
     "--splits", f"{DATA_DIR}/splits.parquet",
     "--split", "test", "--sources", SOURCES, "--n", str(N_TEXTS),
     "--max-words", "5000", "--attacks", "word_typography,plain_typography",
     "--provider", "none", "--out-dir", f"{OUT_DIR}/texts"])


## 4. Score them

In [ ]:
run(["python", "-u", "scripts/eval_robust.py",
     "--attacks", f"{OUT_DIR}/texts/attacks.parquet",
     "--model", f"{MODEL}={MODELS_DIR}/{MODEL}.joblib",
     "--out-dir", f"{OUT_DIR}/{MODEL}"])
print(open(f"{OUT_DIR}/{MODEL}/robust_report.md").read())


The report is in `My Drive/skolfie/reports_v3/typography/`. Tell Claude in the project thread
when this has finished; it reads the report from Drive directly.
